# Hamiltonian simulation with probabilistic Error Amplification (PEA) mitigation

In [1]:
# (C) Copyright IBM 2026.
#
# This code is licensed under the Apache License, Version 2.0. You may
# obtain a copy of this license in the LICENSE.txt file in the root directory
# of this source tree or at http://www.apache.org/licenses/LICENSE-2.0.
#
# Any modifications or derivative works of this code must retain this
# copyright notice, and modified files need to carry a notice indicating
# that they have been altered from the originals.

In [ ]:
import datetime
import json
import numpy as np

from qiskit import *
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from supermarq.benchmarks.hamiltonian_simulation import HamiltonianSimulation

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService, Estimator, Batch, EstimatorOptions

In [ ]:
service = QiskitRuntimeService()

In [ ]:
backend = service.backend("ibm_phoenix")

In [ ]:
sim = AerEstimator(options={'backend_options':{'method': "matrix_product_state"}})

In [ ]:
shots = 2**16

In [ ]:
num_qubits = 10
dt = 0.035
steps = 101
circuits = []
gate_counts = []
trans_gate_counts = []
sim_mags = []

obs = []
for k in range(num_qubits):
    obs.append(SparsePauliOp.from_sparse_list([('Z', [k], 1)], num_qubits=num_qubits))

for kk in range(steps):
    ham = HamiltonianSimulation(num_qubits, dt, dt*kk)
    qc = ham.qiskit_circuit()
    qc = qc.remove_final_measurements(inplace=False)
    gate_counts.append(qc.count_ops())
    circuits.append(qc)
    expvals_ideal = sim.run([(qc, obs)]).result()[0].data.evs
    sim_mag = np.sum(expvals_ideal)/num_qubits
    sim_mags.append(sim_mag)

sim_mags = np.array(sim_mags)

In [ ]:
trans_circuits = transpile(circuits, backend, optimization_level=3)

In [ ]:
trans_gate_counts = []
for circ in trans_circuits:
    trans_gate_counts.append(circ.count_ops())

In [ ]:
options = EstimatorOptions()
# turn on T-REX and ZNE
options.resilience_level = 2

# dynamical decoupling
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XpXm"

# twirling
options.twirling.enable_gates = True
options.twirling.num_randomizations = 1000
options.twirling.shots_per_randomization = 64

# PEA
options.resilience.zne.amplifier = "pea"
options.resilience.zne.noise_factors = [1, 1.6, 1.9, 2.8]

options.resilience.layer_noise_learning.shots_per_randomization = 64
options.resilience.layer_noise_learning.num_randomizations = 50
options.resilience.layer_noise_learning.layer_pair_depths = [0, 6, 16, 32, 64]


In [ ]:
batch = Batch(backend)

In [ ]:
estimator = Estimator(batch, options=options)
estimator.options.default_shots = shots

In [ ]:
jobs = []
for circ in trans_circuits:
    trans_obs = [item.apply_layout(circ.layout) for item in obs]
    job = estimator.run([(circ, trans_obs)])
    jobs.append(job)

In [ ]:
batch.close()

In [ ]:
hw_mags = np.array([np.sum(job.result()[0].data.evs)/num_qubits for job in jobs])

In [ ]:
scores = 1 - np.abs(sim_mags - hw_mags) / 2

In [ ]:
stds = [list(job.result()[0].data.stds) for job in jobs]

In [ ]:
sim_magentizations = [float(item) for item in sim_mags] 
hw_magentizations = [float(item) for item in hw_mags] 
scores = [float(item) for item in scores]
orig_gate_counts = [dict(item) for item in gate_counts]
trans_gate_counts = [dict(item) for item in trans_gate_counts]
cum_time = sum([job.usage() for job in jobs])

In [ ]:
results = {'name': 'hamsim', 'backend': backend.name, 'datetime': batch.details()['started_at'],
           'qubits': num_qubits, 'dt':dt, 'steps':steps, 'shots': shots, 'runtime': cum_time,
           'orig_gate_counts': orig_gate_counts, 'trans_gate_counts': trans_gate_counts,
           'sim_magnetizations': sim_magentizations, 'hw_magnetizations':hw_magentizations, 'scores':scores,
        'stds':stds}

In [ ]:
with open(f'hamsim__results_{num_qubits}Q_PEA.json', 'w') as fp:
    json.dump(results, fp)